In [1]:
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd


In [2]:
bbc_data=pd.read_csv("bbc_news.csv")

In [3]:
bbc_data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [4]:
titles=pd.DataFrame(bbc_data['title'])

In [5]:
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


Lowercase


In [6]:
# for t in titles:
#     t.lower()
titles['lowercase']=titles['title'].str.lower()

In [11]:
en_stopword=stopwords.words('english')
titles['no_stopwords']=titles['lowercase'].apply(lambda x:' '.join([word for word in x.split() if word not in en_stopword]))

In [12]:
titles['no_stopwords_no_punctuation']=titles.apply(lambda x:re.sub(r'[^\w\s]',' ',x['no_stopwords']),axis=1)

In [16]:
titles['tokens_raw'] = titles.apply(lambda x: word_tokenize(x['title']),axis=1)
titles['tokens_clean'] = titles.apply(lambda x: word_tokenize(x['no_stopwords_no_punctuation']),axis=1)

In [21]:
lem=WordNetLemmatizer()
titles['tokens_lemmetized']=titles['tokens_clean'].apply(lambda x: [lem.lemmatize(token, pos='v')  for token in x])

In [22]:
titles.head()

,title,lowercase,no_stopwords,no_stopwords_no_punctuation,tokens_raw,tokens_clean,tokens_lemmetized
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political t...,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, react, uk, politica..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new off grid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, off, grid, c...","[ration, energy, nothing, new, off, grid, comm..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanction, russian, oligarchs]"
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, years, queen, 70, second]"


In [29]:
import spacy
nlp = spacy.load("en_core_web_sm")
tokenlist=sum(titles['tokens_raw'],[])
spacy_doc= nlp(" ".join(tokenlist))
pos_df=pd.DataFrame(columns=['token','pos_tag'])


In [31]:
for token in spacy_doc:
    pos_df=pd.concat([pos_df, pd.DataFrame.from_records([{'token':token.text,'pos_tag':token.pos_}])], ignore_index=True)

In [32]:
nerdf=pd.DataFrame(columns=['token','entity'])
for token in spacy_doc.ents:
    nerdf=pd.concat([nerdf, pd.DataFrame.from_records([{'token':token.text,'entity':token.label_}])],ignore_index=
                    True)
   

In [34]:
pos_df

,token,pos_tag
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB
...,...,...
11742,sale,NOUN
11743,scams,NOUN
11744,",",PUNCT
11745,consumers,NOUN


In [35]:
nerdf

,token,entity
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP
...,...,...
1665,Frenkie de Jong,PERSON
1666,Manchester United,PERSON
1667,Barcelona,GPE
1668,Dominic Raab,PERSON
